In [28]:
#from datetime import datetime
from pathlib import Path
notebook_directory = Path.cwd()

#import numpy as np
import pandas as pd
import xlwings as xw

In [29]:
prices_path = (
    notebook_directory.parent.parent
    / "backtesting"
    / "historical prices"
    / "monthly prices.csv"
)

In [30]:

prices_df = pd.read_csv(prices_path, parse_dates=["date"])
prices_df["date"] = pd.to_datetime(prices_df["date"], errors="coerce")

previous_month = (pd.Timestamp.today() - pd.DateOffset(months=1)).to_period("M")

prices_df = prices_df.loc[prices_df["date"].dt.to_period("M").eq(previous_month)].copy()

In [31]:
db_path = (
    notebook_directory.parent
    / "spreadsheets"
    / "2026 Fin Inst Database.xlsx"
)

In [37]:
wb = xw.Book(db_path)
ws = wb.sheets["BOND ETFs"]

db_df = ws.tables["Table100"].range.options(pd.DataFrame, header=1, index=False).value

print(db_df)

                 Sector           Family Symbol  ETF Fee Margin  \
0                   Agg          iShares    AGG   0.0003   None   
1                   Agg         Vanguard    BND   0.0003   None   
2                   Agg           Schwab   SCHZ   0.0003   None   
3                   Agg     State Street   SPAB   0.0003   None   
4           Calif Munis          iShares    CMF   0.0008   None   
5           Calif Munis         Vanguard   VTEC   0.0006   None   
6              Converts     State Street    CWB   0.0040   None   
7              Converts          iShares   ICVT   0.0020   None   
8                 Corps            PIMCO   CORP   0.0020   None   
9                 Corps     State Street   SPBO   0.0003   None   
10                Corps          iShares   USIG   0.0004   None   
11                Corps         Vanguard    VTC   0.0003   None   
12           EM Gov -US          iShares    EMB   0.0039   None   
13           EM Gov -US         Vanguard   VWOB   0.0015   Non

In [ ]:
div_month = (pd.Timestamp.today() - pd.DateOffset(months=1)).strftime("%Y%m")

div_column = f"Div {div_month}"
symbols = prices_df.columns.drop("date").tolist()

symbol_list = db_df['Symbol'].to_list()

for sym in symbols:
    if sym in symbol_list:
        dividends = db_df.loc[db_df["Symbol"].eq(sym), div_column].to_list()

        div_amt = float(dividends[0])
    else:
        div_amt = 0

prices_df[f"{sym}*"] = prices_df[sym] - div_amt

AGG
AGG
ANGL
ANGL
BND
BND
BNDX
BNDX
BWX
BWX
CMF
CMF
CORP
CORP
CWB
CWB
EMB
EMB
FALN
FALN
HYG
HYG
HYLB
HYLB
IAGG
IAGG
ICVT
ICVT
IGLB
IGLB
IGOV
IGOV
JNK
JNK
MBB
MBB
MUB
MUB
PFF
PFF
PFFD
PFFD
RSP
SCHH
SCHQ
SCHQ
SCHZ
SCHZ
SCYB
SCYB
SPAB
SPAB
SPBO
SPBO
SPHY
SPHY
SPLB
SPLB
SPMB
SPMB
SPTL
SPTL
SPY
TFI
TFI
TLT
USHY
USHY
USIG
USIG
USRT
VCLT
VCLT
VGLT
VGLT
VMBS
VMBS
VTC
VTC
VTEB
VTEB
VTEC
VTEC
VWOB
VWOB


In [42]:
print(prices_df)

           date    AGG   ANGL    BND   BNDX    BWX    CMF   CORP     CWB  \
1218 2026-07-01  98.50  29.11  73.06  48.27  21.59  57.51  96.33  105.85   
1219 2026-07-02  98.61  29.16  73.11  48.24  21.68  57.58  96.52  104.38   
1220 2026-07-06  98.66  29.18  73.14  48.22  21.67  57.58  96.52  105.33   
1221 2026-07-07  98.21  29.14  72.85  48.02  21.55  57.48  96.03  103.85   
1222 2026-07-08  98.04  29.05  72.70  47.83  21.50  57.33  95.87  104.65   
1223 2026-07-09  98.18  29.09  72.83  48.02  21.52  57.36  95.97  105.62   
1224 2026-07-10  98.08  29.03  72.77  48.08  21.59  57.35  95.79  105.24   
1225 2026-07-13  97.71  28.96  72.50  47.89  21.45  57.33  95.41  103.86   
1226 2026-07-14  98.00  29.03  72.70  47.96  21.59  57.34  95.64  104.14   
1227 2026-07-15  98.14  29.07  72.82  48.03  21.66  57.25  95.90  103.59   
1228 2026-07-16  98.13  29.06  72.81  47.96  21.59  57.12  95.88  101.54   
1229 2026-07-17  98.20  29.05  72.86  47.93  21.56  57.11  95.93  101.49   
1230 2026-07

In [ ]:
anchor_path = (
    notebook_directory.parent
    / "spreadsheets"
    / "2026 Group Trading Inputs.xlsx"
)

In [ ]:

wb = xw.Book(anchor_path)
ws = wb.sheets["FI INPUTs"]

anchor_df = ws.tables["fi_inputs"].range.options(pd.DataFrame, header=1, index=False).value

In [ ]:
for sym in symbols:
    anchor = anchor_df.loc[anchor_df["my_fi_name"].eq(sym), "anchor_fi"].iloc[0]
    prices_df[f"{sym}**"] = prices_df[f'{anchor}*'] / prices_df[f'{sym}*']
    avg_ratio = prices_df[f"{sym}**"].mean()
    anchor_df.loc[anchor_df['my_fi_name' == sym], "multiplier"] = avg_ratio

In [ ]:
print(anchor_df)

           date   SPMB   VMBS    MBB  SPMB*  VMBS*   MBB*  MBB* / SPMB*  \
1218 2026-07-01  22.18  46.57  94.02  22.17  46.56  94.01      4.240415   
1219 2026-07-02  22.20  46.61  94.13  22.19  46.60  94.12      4.241550   
1220 2026-07-06  22.23  46.62  94.28  22.22  46.61  94.27      4.242574   
1221 2026-07-07  22.13  46.46  93.84  22.12  46.45  93.83      4.241863   
1222 2026-07-08  22.11  46.37  93.69  22.10  46.36  93.68      4.238914   
1223 2026-07-09  22.13  46.47  93.82  22.12  46.46  93.81      4.240958   
1224 2026-07-10  22.12  46.44  93.72  22.11  46.43  93.71      4.238354   
1225 2026-07-13  22.02  46.24  93.26  22.01  46.23  93.25      4.236711   
1226 2026-07-14  22.12  46.46  93.67  22.11  46.45  93.66      4.236092   
1227 2026-07-15  22.14  46.50  93.83  22.13  46.49  93.82      4.239494   
1228 2026-07-16  22.13  46.45  93.76  22.12  46.44  93.75      4.238246   
1229 2026-07-17  22.14  46.50  93.78  22.13  46.49  93.77      4.237235   
1230 2026-07-20  22.08  4